# drop unknown EDA

In [766]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import SMOTE

df = pd.read_csv("./bank-additional/bank-additional/bank-additional-full.csv",delimiter=";")
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


#### drop yg aneh2

In [767]:
# df.drop(['nr.employed','euribor3m','cons.conf.idx','cons.price.idx','emp.var.rate'],axis=1,inplace=True)
df.drop(['emp.var.rate','contact','previous','housing','loan'],axis=1,inplace=True)




#### genapin duration

In [768]:
df['duration'] = (df['duration']/60).round()

#### drop column

In [769]:
df.drop(['day_of_week','default','pdays'],axis=1,inplace=True)
df.head()

,age,job,marital,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,may,2.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,may,3.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,may,5.0,1,nonexistent,93.994,-36.4,4.857,5191.0,no


#### drop unknown

In [770]:
df = df[df['duration'] <= 35]

In [771]:
df = df[df['education'] != 'unknown']


In [772]:
df = df[df['campaign'] <= 30]

In [773]:
# df = df[df['housing'] != 'unknown']

In [774]:
df = df[df['job'] != 'unknown']

In [775]:
df = df[df['marital'] != 'unknown']

#### feature age = bining & ordinal

In [776]:
'''
min_age = df['age'].min()   # 17
max_age = df['age'].max()   # 98

# 5‑year buckets: [17–21], [22–26], … , [87–91] (last bucket includes all >90)
bin_edges = list(range(min_age, max_age + 1, 5))      # e.g. [17, 22, 27, ..., 93]
# add an upper bound that is bigger than any age so the last value falls in
bin_edges.append(float('inf'))

# Labels for readability (optional)
labels = [f'{l}-{r-1}' for l, r in zip(bin_edges[:-2], bin_edges[1:-1])]
labels.append(f'{bin_edges[-2]}+')

# ── Apply cut ───────────────────────────────────────────
df['age_bin'] = pd.cut(df['age'],
                       bins=bin_edges,
                       labels=labels,
                       right=False,          # left‑inclusive [l,r)
                       include_lowest=True)  # include the very first value

# ── Ordinal encoding (1 = first bin, 2 = second …) ───────
df['age_ordinal'] = df['age_bin'].cat.codes + 1   # .codes gives 0‑based int

df.head()
'''


"\nmin_age = df['age'].min()   # 17\nmax_age = df['age'].max()   # 98\n\n# 5‑year buckets: [17–21], [22–26], … , [87–91] (last bucket includes all >90)\nbin_edges = list(range(min_age, max_age + 1, 5))      # e.g. [17, 22, 27, ..., 93]\n# add an upper bound that is bigger than any age so the last value falls in\nbin_edges.append(float('inf'))\n\n# Labels for readability (optional)\nlabels = [f'{l}-{r-1}' for l, r in zip(bin_edges[:-2], bin_edges[1:-1])]\nlabels.append(f'{bin_edges[-2]}+')\n\n# ── Apply cut ───────────────────────────────────────────\ndf['age_bin'] = pd.cut(df['age'],\n                       bins=bin_edges,\n                       labels=labels,\n                       right=False,          # left‑inclusive [l,r)\n                       include_lowest=True)  # include the very first value\n\n# ── Ordinal encoding (1\u202f=\u202ffirst bin, 2\u202f=\u202fsecond …) ───────\ndf['age_ordinal'] = df['age_bin'].cat.codes + 1   # .codes gives 0‑based int\n\ndf.head()\n"

In [777]:
# drop obsolete age feature :
# df.drop(['age','age_bin'],axis = 1,inplace=True)

In [778]:
# Convert y from yes/no to 0/1
df['y'] = df['y'].map({'yes': 1, 'no': 0})

#### contact binary encoder

In [779]:
from category_encoders import BinaryEncoder,OneHotEncoder,TargetEncoder


#encoder = BinaryEncoder(cols=['contact'])
#df = encoder.fit_transform(df)

#### education OHE

In [780]:
ohe = TargetEncoder(cols=['education'])
df = ohe.fit_transform(df,df['y'])

df.head()

,age,job,marital,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,0.102383,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
1,57,services,married,0.108217,may,2.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
2,37,services,married,0.108217,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,0.082264,may,3.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
4,56,services,married,0.108217,may,5.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0


#### housing binary

In [781]:
#encoder2 = BinaryEncoder(cols=['housing'])
#df = encoder2.fit_transform(df)

df.head()

,age,job,marital,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,0.102383,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
1,57,services,married,0.108217,may,2.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
2,37,services,married,0.108217,may,4.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
3,40,admin.,married,0.082264,may,3.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0
4,56,services,married,0.108217,may,5.0,1,nonexistent,93.994,-36.4,4.857,5191.0,0


#### job OHE,loan OHE, marital OHE,month OHE, poutcome OHE


contact,previous,housing,loan

In [782]:
encoder3 = TargetEncoder(cols=['job'])
df = encoder3.fit_transform(df,df['y'])

#encoder4 = BinaryEncoder(cols=['loan'])
#df = encoder4.fit_transform(df)

encoder5 = TargetEncoder(cols=['marital'])
df = encoder5.fit_transform(df,df['y'])

encoder6 = TargetEncoder(cols=['month'])
df = encoder6.fit_transform(df,df['y'])

encoder7 = TargetEncoder(cols=['poutcome'])
df = encoder7.fit_transform(df,df['y'])



df.head()


,age,job,marital,education,month,duration,campaign,poutcome,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,0.097826,0.100722,0.102383,0.064292,4.0,1,0.087561,93.994,-36.4,4.857,5191.0,0
1,57,0.079811,0.100722,0.108217,0.064292,2.0,1,0.087561,93.994,-36.4,4.857,5191.0,0
2,37,0.079811,0.100722,0.108217,0.064292,4.0,1,0.087561,93.994,-36.4,4.857,5191.0,0
3,40,0.128736,0.100722,0.082264,0.064292,3.0,1,0.087561,93.994,-36.4,4.857,5191.0,0
4,56,0.079811,0.100722,0.108217,0.064292,5.0,1,0.087561,93.994,-36.4,4.857,5191.0,0


#### train test SPLIT

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from imblearn.over_sampling import SMOTE

           
X = df.drop('y', axis=1)  
y = df['y']   

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y            
)

'''
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
'''
'''
from sklearn.preprocessing import MinMaxScaler
minmax = MinMaxScaler()
X_train = minmax.fit_transform(X_train)
X_test = minmax.transform(X_test)
'''

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

from collections import Counter

print(Counter(y_train_res))

Counter({0: 27823, 1: 27823})


#### train-test

In [784]:

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = DecisionTreeClassifier(random_state=42)   # reproducible tree structure
clf.fit(X_train_res, y_train_res)

y_pred = clf.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

import pandas as pd 

feature_importances = pd.Series(clf.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)

print("\nFeature Importances:")
print(feature_importances)


Accuracy: 0.8921

Classification Report:
              precision    recall  f1-score   support

           0       0.94      0.93      0.94      6957
           1       0.51      0.55      0.53       867

    accuracy                           0.89      7824
   macro avg       0.73      0.74      0.73      7824
weighted avg       0.90      0.89      0.89      7824


Confusion Matrix:
[[6504  453]
 [ 391  476]]

Feature Importances:
duration          0.491933
euribor3m         0.129908
month             0.093380
nr.employed       0.071663
age               0.048880
education         0.038446
job               0.036942
poutcome          0.025618
cons.price.idx    0.019275
campaign          0.018170
marital           0.013087
cons.conf.idx     0.012698
dtype: float64


contact,previous,housing,loan

In [785]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

# Dynamic range for a hyper‑parameter (example: max_depth 1–100)
max_depth_range = list(range(1, 101))

param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 4, 6, 8],
    'min_samples_leaf': [1, 2, 3, 4]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid.fit(X_train_res, y_train_res)
clf_best = grid.best_estimator_

y_pred = clf_best.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

feature_importances = pd.Series(clf_best.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)

print("\nFeature Importances:")
print(feature_importances)


Accuracy: 0.8956

Classification Report:
              precision    recall  f1-score   support

           0       0.95      0.93      0.94      6957
           1       0.53      0.60      0.56       867

    accuracy                           0.90      7824
   macro avg       0.74      0.77      0.75      7824
weighted avg       0.90      0.90      0.90      7824


Confusion Matrix:
[[6486  471]
 [ 346  521]]

Feature Importances:
duration          0.515524
euribor3m         0.125002
month             0.097787
nr.employed       0.075682
education         0.037111
age               0.035616
job               0.033093
poutcome          0.026754
cons.conf.idx     0.023960
campaign          0.013322
marital           0.010434
cons.price.idx    0.005714
dtype: float64
